# 0. Goals 05_Modeling V3 - Reduce Feature

- Transform Categorical Column to Numerical
- Re Modelling Using New Feature
- Reduce Feature Based on Result 04_Modeling V2

Detail Feature Importance:
- Mutual Importance : [Bucket Amount, Bucket V14, Bucket V9, Bucket V7, Bucket 12, Bucket V1, Bucket V10, Bucket V24, Bucket V3, Bucket Amount]
- Extra Trees Classifier Feature Importance : [V14, V17, V12, V11, V16, V10, V4. V3, V18, V9]
- Random Forest Classifier Feature Importance : [V14, V10, V17, V4, V12, V11, V3. V16, V7, V2]
- CatBoost Classifier Feature Importance : [V26, V4, V1, V14, V12, V8, Amount. V15, Time, V20]

In [52]:
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.feature_selection import mutual_info_classif
from catboost import CatBoostClassifier

In [53]:
path_data_train = r'..\Data\result\03_Exploratory_Data_Analysis_Train.csv'

In [54]:
df_train = pd.read_csv(path_data_train)

In [55]:
df_train = df_train[['Bucket_Amount', 'Bucket_V14', 'Bucket_V9', 'Bucket_V7', 'Bucket_V12', 'Bucket_V1', 'Bucket_V10', 'Bucket_V24', 'Bucket_V3', 'V14', 'V17', 'V12', 'V11', 'V16', 'V10', 'V4', 'V3', 'V18', 'V9', 'V26', 'V1', 'V8', 'Amount', 'V15', 'Time', 'V20', 'Class']]

# 1. Transform Categorical Column to Numerical

In [56]:
# Cek kolom yang non numerical untuk di encoding

df_train.select_dtypes(include=['str']).info()

<class 'pandas.DataFrame'>
RangeIndex: 255387 entries, 0 to 255386
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Bucket_Amount  255387 non-null  str  
 1   Bucket_V14     255387 non-null  str  
 2   Bucket_V9      255387 non-null  str  
 3   Bucket_V7      255387 non-null  str  
 4   Bucket_V12     255387 non-null  str  
 5   Bucket_V1      255387 non-null  str  
 6   Bucket_V10     255387 non-null  str  
 7   Bucket_V24     255387 non-null  str  
 8   Bucket_V3      255387 non-null  str  
dtypes: str(9)
memory usage: 17.5 MB


In [19]:
# Karena kolom 'bucket delta time' dan kolom 'bucket percent amount' itu memperahtikan urutan (ordinal) maka pakainya ordinal encoder

model_encoder = OrdinalEncoder()
df_train[df_train.select_dtypes(include=['str']).columns.tolist()] = model_encoder.fit_transform(df_train[df_train.select_dtypes(include=['str']).columns.tolist()])

In [20]:
# export model encodernya pakai joblib

joblib.dump(model_encoder, r'..\Output\Model\model_encoder_ordinal.pkl')

['..\\Output\\Model\\model_encoder_ordinal.pkl']

In [21]:
df_train

,Bucket_Amount,Bucket_V14,Bucket_V9,Bucket_V7,Bucket_V12,Bucket_V1,Bucket_V10,Bucket_V24,Bucket_V3,V14,...,V18,V9,V26,V1,V8,Amount,V15,Time,V20,Class
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-0.311169,...,0.025791,0.363787,-0.189115,-1.359807,0.098698,149.62,1.468177,0.0,0.251412,0
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-0.143772,...,-0.183361,-0.255425,0.125895,1.191857,0.085102,2.69,0.635558,0.0,-0.069083,0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-0.165946,...,-0.121359,-1.514654,-0.139097,-1.358354,0.247676,378.66,2.345865,1.0,0.524980,0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-0.287924,...,1.965775,-1.387024,-0.221929,-0.966272,0.377436,123.50,-0.631418,1.0,-0.208038,0
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.119670,...,-0.038195,0.817739,0.502292,-1.158233,-0.270533,69.99,0.175121,2.0,0.408542,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255382,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.764766,...,-0.024656,0.005444,-0.485768,2.006662,-0.236439,9.65,-1.165311,157630.0,-0.295280,0
255383,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-0.101518,...,0.113980,1.388603,0.570915,1.954214,-0.016540,28.75,0.440551,157630.0,-0.203095,0
255384,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.505877,...,-0.337455,0.665536,-0.622943,1.849530,-0.222235,149.98,1.742599,157631.0,-0.410845,0
255385,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.516138,...,-0.231017,0.352145,0.199448,2.043229,-0.149102,0.99,-0.602179,157631.0,-0.225050,0


# Split Dataset

In [22]:
X = df_train.drop(columns='Class')
y = df_train['Class']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [24]:
train = X_train.copy()
train['Class'] = y_train

test = X_test.copy()
test['Class'] = y_test

# Re-Train Model

In [25]:
# Dengan AUPRC 0.849
best_ETC_params = {
    'n_estimators'        : 296, 
    'max_depth'           : 37, 
    'min_samples_split'   : 11, 
    'min_samples_leaf'    : 3, 
    'criterion'           : 'entropy', 
    'class_weight'        : None
}

# Dengan AUPRC 0.852
best_RFC_params = {
    'n_estimators'        : 691, 
    'max_depth'           : 23, 
    'min_samples_split'   : 4, 
    'min_samples_leaf'    : 4, 
    'criterion'           : 'log_loss', 
    'class_weight'        : 'balanced_subsample', 
    'bootstrap'           : False
}

# Dengan AUPRC 0.853
best_CB_params = {
    'iterations': 594, 
    'depth': 9, 
    'learning_rate': 0.010639915618971553, 
    'l2_leaf_reg': 1.1199438541669806, 
    'random_strength': 0.8875933831253247
}

In [26]:
etc_model_v3 = ExtraTreesClassifier(
    **best_ETC_params,
    random_state=42,
    n_jobs=-1
)

rfc_model_v3 = RandomForestClassifier(
    **best_RFC_params,
    random_state=42,
    n_jobs=-1
)

cb_model_v3 = CatBoostClassifier(
    **best_CB_params,
    verbose=False,
    thread_count=-1
)

## Extra Trees Classifier

In [27]:
start_etc_train = time.perf_counter()

etc_model_v3.fit(X_train, y_train)

end_etc_train = time.perf_counter()

training_etc_model_time = end_etc_train - start_etc_train

### Test

In [28]:
start_etc_test = time.perf_counter()

etc_pred = etc_model_v3.predict_proba(X_test)[:, 1]

end_etc_test = time.perf_counter()

test_etc_model_time = end_etc_test - start_etc_test

In [31]:
etc_v3_auprc = average_precision_score(y_test, etc_pred)

In [32]:
print(f"Training time Extra Trees Classifier Model \t: {training_etc_model_time:.4f} seconds")
print(f"Test (Inference) time Extra Trees Classifier Model \t: {test_etc_model_time:.4f} seconds")

print()
print('Score AUPRC Extra Trees Classifier Model \t:',  etc_v3_auprc)

Training time Extra Trees Classifier Model 	: 15.6287 seconds
Test (Inference) time Extra Trees Classifier Model 	: 0.2102 seconds

Score AUPRC Extra Trees Classifier Model 	: 0.8764410522019193


## Random Forest Classifier

In [34]:
start_rfc_train = time.perf_counter()

rfc_model_v3.fit(X_train, y_train)

end_rfc_train = time.perf_counter()

training_rfc_model_time = end_rfc_train - start_rfc_train

### Test

In [35]:
start_rfc_test = time.perf_counter()

rfc_pred = rfc_model_v3.predict_proba(X_test)[:, 1]

end_rfc_test = time.perf_counter()

test_rfc_model_time = end_rfc_test - start_rfc_test

In [36]:
rfc_v3_auprc = average_precision_score(y_test, rfc_pred)

In [37]:
print(f"Training time Random Forest Classifier Model: {training_rfc_model_time:.4f} seconds")
print(f"Test (Inference) time Random Forest Classifier Model: {test_rfc_model_time:.4f} seconds")

print()
print('Score AUPRC Random Forest Classifier Model \t:',  rfc_v3_auprc)

Training time Random Forest Classifier Model: 110.0970 seconds
Test (Inference) time Random Forest Classifier Model: 0.3899 seconds

Score AUPRC Random Forest Classifier Model 	: 0.894465741072201


## CatBoost Classifier

In [39]:
start_cb_train = time.perf_counter()

cb_model_v3.fit(X_train, y_train)

end_cb_train = time.perf_counter()

training_cb_model_time = end_cb_train - start_cb_train

### Test

In [40]:
start_cb_test = time.perf_counter()

cb_pred = cb_model_v3.predict_proba(X_test)[:, 1]

end_cb_test = time.perf_counter()

test_cb_model_time = end_cb_test - start_cb_test

In [41]:
cb_v3_auprc = average_precision_score(y_test, cb_pred)

In [42]:
print(f"Training time CatBoost Classifier Model: {training_cb_model_time:.4f} seconds")
print(f"Test (Inference) time CatBoost Classifier Model: {test_cb_model_time:.4f} seconds")

print()
print('Score AUPRC CatBoost Classifier Model \t:',  cb_v3_auprc)

Training time CatBoost Classifier Model: 24.6050 seconds
Test (Inference) time CatBoost Classifier Model: 0.0371 seconds

Score AUPRC CatBoost Classifier Model 	: 0.8958554707942097


# Export Model V3

In [50]:
cb_model_v3.save_model("../Output/Model/model_v3_catboost.cbm")
joblib.dump(etc_model_v3,"../Output/Model/model_v3_extra_trees.pkl")
joblib.dump(rfc_model_v3,"../Output/Model/model_v3_random_forest.pkl")

['../Output/Model/model_v3_random_forest.pkl']

# Summary V1 vs V2

Berikut merupakan perbandingan Performa dan Detail model V1 vs V2

<h3 style="text-align: center;">TABLE TRAIN</h3>
<table>
    <tr>
        <th rowspan="2">Model</th>
        <th colspan="3">AUPRC</th>
        <th colspan="3">Training</th>
        <th colspan="3">Validation</th>
      </tr>
    <tr>
        <th>V1</th>
        <th>V2</th>
        <th>V3</th>
        <th>V1</th>
        <th>V2</th>
        <th>V3</th>
        <th>V1</th>
        <th>V2</th>
        <th>V3</th>
    </tr>
    <tr>
        <td>Extra Trees</td>
        <td>0.81</td>
        <td>0.87</td>
        <td>0.87</td>
        <td>30.5s</td>
        <td>18.5s</td>
        <td>15.6s</td>
        <td>0.74s</td>
        <td>0.2s</td>
        <td>0.2s</td>
    </tr>
    <tr>
        <td>Random Forest</td>
        <td>0.82</td>
        <td>0.89</td>
        <td>0.89</td>
        <td>226.5s</td>
        <td>181.9s</td>
        <td>110.0s</td>
        <td>0.46s</td>
        <td>1.0s</td>
        <td>0.38s</td>
    </tr>
    <tr>
        <td>CatBoost</td>
        <td>0.81</td>
        <td>0.89</td>
        <td>0.89</td>
        <td>29.8s</td>
        <td>144.2s</td>
        <td>24.6s</td>
        <td>0.05s</td>
        <td>0.04s</td>
        <td>0.03s</td>
    </tr>
</table>

Berdasarkan table diatas, di rekomendasikan melakukan reduksi feature serta menggunakan model CatBoost Classifier. Hal ini dikarenakan:
- Menjadi peringkat 1 dari segi performa
- Dengan total data 200 ribuan, waktu yang dibutuhkan untuk melakukan maintenance model terbilang cukup cepat hanya 25 detik (lebih lama dibandingkan Extra Trees Model dengan selisih lebih lama 9 detik namun unggul performa sejumlah 0.02 poin)
- Waktu yang diperlukan untuk melakukan predict 1 data baru hanya membutuhkan waktu 0.03 detik. Hal ini sangat cocok untuk bisnis yang memerlukan kecepatan dalam pemeriksaan apakah suatu transaksi merupakan transaksi fraud atau bukan